In [18]:

#TypedDict
from ..structure_output import *

#结构化输出
load_dotenv(override=True)

DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')
model=init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek-v4-flash',
    model_provider='deepseek',
    #flash思考模式不支持结构化输出
    extra_body={"thinking":{"type":"disabled"}}
)
#Annotated类似于Field,其中...表示该属性必须存在但可以没有值
class movieDict(TypedDict):
    title:Annotated[str,...,'电影名字']
    year: Annotated[int,...,'电影上映年份']
    director:Annotated[str,...,'导演']
    score:Annotated[float,...,'平分']
class movieDict_2(TypedDict):
    title:Annotated[str,...,'电影名字']
    year: Annotated[int,'电影上映年份']
    director:Annotated[str,...,'导演']
    score:Annotated[float,'平分']

In [19]:
movie_model=model.with_structured_output(movieDict)
movie_model2=model.with_structured_output(movieDict_2)
movie=movie_model.invoke('根据下面的话提取电影信息,不包含的信息可以留空：盗梦空间是克里斯托弗·诺兰导演的电影')
movie2=movie_model2.invoke('根据下面的话提取电影信息,不包含的信息可以留空：盗梦空间是克里斯托弗·诺兰导演的电影')
print(type(movie))
print(movie)
print(movie2)

<class 'dict'>
{'title': '盗梦空间', 'year': 0, 'director': '克里斯托弗·诺兰', 'score': 0}
{'title': '盗梦空间', 'director': '克里斯托弗·诺兰'}


In [20]:
#json_schema
"""
使用 JSON Schema 定义嵌套结构
"""
# 1. 定义嵌套的 JSON Schema
project_schema = {
    "title": "MovieInfo",
    "description": "包含电影标题、上映年份、导演、演员和评分的电影对象",
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "电影标题"},
        "year": {"type": "integer", "description": "上映年份"},
        "director": {"type": "string", "description": "导演"},
        "cast": {  # 定义嵌套数组
            "type": "array",
            "description": "演员列表",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "演员姓名"},
                    "role": {"type": "string", "description": "演员角色"}
                },
                "required": ["name", "role"]
            }
        },
        "rating": {"type": "number", "description": "评分（10分制）"}
    },
    "required": ["title", "year", "director", "cast", "rating"]
}
# 绑定 JSON Schema 到模型,method为结构化输出的方式
structured_model = model.with_structured_output(project_schema,method='json_schema')
response = structured_model.invoke("生成一个关于《星际穿越》的电影信息，包含导演、演员、评分")
print(type(response))
rprint(response)

<class 'dict'>


{
    'title': '星际穿越',
    'year': 2014,
    'director': '克里斯托弗·诺兰',
    'cast': [
        {'name': '马修·麦康纳', 'role': '库珀'},
        {'name': '安妮·海瑟薇', 'role': '布兰德博士'},
        {'name': '杰西卡·查斯坦', 'role': '墨菲（成年）'},
        {'name': '迈克尔·凯恩', 'role': '布兰德教授'},
        {'name': '麦肯吉·弗依', 'role': '墨菲（少年）'},
        {'name': '蒂莫西·柴勒梅德', 'role': '汤姆（少年）'},
        {'name': '比尔·欧文', 'role': '阿米莉亚·布兰德'},
        {'name': '约翰·利思戈', 'role': '唐纳德'}
    ],
    'rating': 9.4
}

In [21]:
#dataclass
@dataclass
class company:
    address:str
    phone:str
    name:str
company_model=model.with_structured_output(company)
company=company_model.invoke('介绍一下百度公司')
print(type(company))
print(company)

<class 'dict'>
{'name': '百度', 'address': '中国北京市海淀区上地十街10号百度大厦', 'phone': '+86 10 5992 8888'}
